In [ ]:
# ============================================================
# 01_data_audit.ipynb — AUDIT COMPLET DES DONNÉES
# Objectif : observer, documenter, ne rien modifier
# ============================================================

import sys
sys.path.append('..')
from src.utils import *

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 1 — CHARGEMENT DES TABLES BRUTES")
print("   Source : data/raw/ — lecture seule")
print("="*60 + "\n")
# -------------------------------------------------------

orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments  = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews   = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products  = pd.read_csv('../data/raw/olist_products_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
sellers   = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
geo       = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
transl    = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Résumé rapide du chargement
print(f"{'Table':<15} | {'Lignes':>8} | {'Colonnes':>8}")
print("-" * 40)
for name, df in [('orders',orders),('items',items),
                 ('payments',payments),('reviews',reviews),
                 ('products',products),('customers',customers),
                 ('sellers',sellers),('geo',geo),('transl',transl)]:
    print(f"  {name:<15} | {len(df):>8} | {df.shape[1]:>8}")

print("\n✓ Toutes les tables chargées — aucune modification")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 2 — AUDIT STRUCTURE : TOUTES LES TABLES")
print("="*60 + "\n")
# -------------------------------------------------------


# Affiche : dimensions, types colonnes, aperçu 2 lignes
for name, df in [('orders',orders),('items',items),
                 ('payments',payments),('reviews',reviews),
                 ('products',products),('customers',customers),
                 ('sellers',sellers),('geo',geo),('transl',transl)]:
    audit_table(name, df)


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 3 — RAPPORT DES VALEURS MANQUANTES")
print("="*60 + "\n")
# -------------------------------------------------------

# Affiche : nulls par colonne + pourcentage + flag si >5%
rapport_nulls({
    'orders':    orders,
    'items':     items,
    'payments':  payments,
    'reviews':   reviews,
    'products':  products,
    'customers': customers,
    'sellers':   sellers,
    'geo':       geo,
    'transl':    transl
})


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 4 — AUDIT DOUBLONS : TOUTES LES TABLES")
print("="*60 + "\n")
# -------------------------------------------------------

print(f"{'Table':<15} | {'Doublons':>10} | {'% total':>8} | {'Statut':>10}")
print("-" * 55)

for name, df in [('orders',orders),('items',items),
                 ('payments',payments),('reviews',reviews),
                 ('products',products),('customers',customers),
                 ('sellers',sellers),('geo',geo),('transl',transl)]:
    n_dup = df.duplicated().sum()
    pct   = n_dup / len(df) * 100
    statut = "✓ OK" if n_dup == 0 else "⚠️ À traiter"
    print(f"  {name:<15} | {n_dup:>10} | {pct:>7.2f}% | {statut:>10}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 5 — AUDIT TYPES : CLASSIFICATION PAR COLONNE")
print("="*60 + "\n")
# -------------------------------------------------------

# Classification manuelle des colonnes par type statistique
# Cours 3 §2 : le type pandas ≠ le type statistique
# Ex : review_score est int64 mais c'est une variable ORDINALE

classification = {
    'orders': {
        'order_id':                    ('object',  'Identifiant — clé primaire'),
        'customer_id':                 ('object',  'Identifiant — clé étrangère'),
        'order_status':                ('object',  'Nominale — catégories sans ordre'),
        'order_purchase_timestamp':    ('object',  'Continue — date à convertir'),
        'order_approved_at':           ('object',  'Continue — date à convertir'),
        'order_delivered_carrier_date':('object',  'Continue — date à convertir'),
        'order_delivered_customer_date':('object', 'Continue — date à convertir'),
        'order_estimated_delivery_date':('object', 'Continue — date à convertir'),
    },
    'items': {
        'order_id':          ('object',  'Identifiant — clé étrangère'),
        'order_item_id':     ('int64',   'Discrète — numéro article dans commande'),
        'product_id':        ('object',  'Identifiant — clé étrangère'),
        'seller_id':         ('object',  'Identifiant — clé étrangère'),
        'shipping_limit_date':('object', 'Continue — date à convertir'),
        'price':             ('float64', 'Continue — à analyser outliers IQR'),
        'freight_value':     ('float64', 'Continue — à analyser outliers IQR'),
    },
    'payments': {
        'order_id':              ('object',  'Identifiant — clé étrangère'),
        'payment_sequential':    ('int64',   'Discrète — ordre des paiements'),
        'payment_type':          ('object',  'Nominale — à convertir category'),
        'payment_installments':  ('int64',   'Discrète — nombre de mensualités'),
        'payment_value':         ('float64', 'Continue — montant payé'),
    },
    'reviews': {
        'review_id':              ('object', 'Identifiant — clé primaire'),
        'order_id':               ('object', 'Identifiant — clé étrangère'),
        'review_score':           ('int64',  'Ordinale — 1 à 5, ordre logique'),
        'review_comment_title':   ('object', 'Texte — optionnel, nulls normaux'),
        'review_comment_message': ('object', 'Texte — optionnel, nulls normaux'),
        'review_creation_date':   ('object', 'Continue — date à convertir'),
        'review_answer_timestamp':('object', 'Continue — date à convertir'),
    },
    'products': {
        'product_id':                  ('object',  'Identifiant — clé primaire'),
        'product_category_name':       ('object',  'Nominale — à imputer + category'),
        'product_name_lenght':         ('float64', 'Discrète — longueur nom'),
        'product_description_lenght':  ('float64', 'Discrète — longueur description'),
        'product_photos_qty':          ('float64', 'Discrète — nombre de photos'),
        'product_weight_g':            ('float64', 'Continue — poids en grammes'),
        'product_length_cm':           ('float64', 'Continue — dimension'),
        'product_height_cm':           ('float64', 'Continue — dimension'),
        'product_width_cm':            ('float64', 'Continue — dimension'),
    }
}

for table, cols in classification.items():
    print(f"--- {table.upper()} ---")
    print(f"  {'Colonne':<35} {'Type pandas':<12} {'Type statistique'}")
    print(f"  {'-'*75}")
    for col, (dtype, stat_type) in cols.items():
        print(f"  {col:<35} {dtype:<12} {stat_type}")
    print()


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 6 — AUDIT SPÉCIFIQUE : orders")
print("   Diagnostic des nulls sur les dates de livraison")
print("="*60 + "\n")
# -------------------------------------------------------

print("--- Nulls par colonne ---")
print(orders.isnull().sum())

print("\n--- Statut des commandes SANS date de livraison ---")
print(orders[orders['order_delivered_customer_date'].isnull()]
      ['order_status'].value_counts())

delivered_sans_date = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]
print(f"\n⚠️  Commandes 'delivered' sans date : {len(delivered_sans_date)}")
print(f"   → Erreurs réelles à supprimer dans 02_data_cleaning")

print(f"\n--- Distribution order_status ---")
print(orders['order_status'].value_counts())


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 7 — AUDIT SPÉCIFIQUE : products")
print("   Diagnostic des nulls et impact sur commandes")
print("="*60 + "\n")
# -------------------------------------------------------

print("--- Nulls par colonne ---")
print(products.isnull().sum())

sans_categorie = products[products['product_category_name'].isnull()]
print(f"\nProduits sans catégorie : {len(sans_categorie)}")

produits_ids = sans_categorie['product_id'].tolist()
commandes_affectees = items[items['product_id'].isin(produits_ids)]
print(f"Commandes affectées    : {len(commandes_affectees)}")
print(f"→ Suppression impossible — imputer 'unknown' dans 02_data_cleaning")

print(f"\n--- Produits sans dimensions physiques ---")
dims = ['product_weight_g','product_length_cm',
        'product_height_cm','product_width_cm']
for col in dims:
    print(f"  {col:<30} {products[col].isnull().sum()} nulls")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 8 — AUDIT SPÉCIFIQUE : reviews")
print("   Confirmer que les nulls commentaires sont intentionnels")
print("="*60 + "\n")
# -------------------------------------------------------

print("--- Nulls par colonne ---")
print(reviews.isnull().sum())

print(f"\nreview_score nulls : {reviews['review_score'].isnull().sum()}")
print(f"→ 0 null sur le score = normal, commentaire est optionnel")

print(f"\n--- Distribution review_score ---")
print(reviews['review_score'].value_counts().sort_index())

total = len(reviews)
score5 = (reviews['review_score'] == 5).sum()
print(f"\nScore 5 : {score5} ({score5/total*100:.1f}%) "
      f"→ dataset biaisé positivement")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 9 — AUDIT SPÉCIFIQUE : items")
print("   Statistiques prix et fret — aperçu avant outliers")
print("="*60 + "\n")
# -------------------------------------------------------

print("--- Statistiques prix ---")
print(items['price'].describe())

print("\n--- Statistiques freight_value ---")
print(items['freight_value'].describe())

# Ratio items/orders — combien d'articles par commande en moyenne
ratio = len(items) / len(orders)
print(f"\nRatio items/orders : {ratio:.2f} articles par commande en moyenne")

# Cas fret > prix — aperçu avant traitement
fret_sup = (items['freight_value'] > items['price']).sum()
print(f"Cas fret > prix    : {fret_sup} ({fret_sup/len(items)*100:.1f}%)")
print(f"→ Anomalie bivariée à flaguer dans 02_data_cleaning")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 10 — AUDIT SPÉCIFIQUE : customers")
print("   Problème customer_id vs customer_unique_id")
print("="*60 + "\n")
# -------------------------------------------------------

n_ids    = customers['customer_id'].nunique()
n_unique = customers['customer_unique_id'].nunique()
diff     = n_ids - n_unique

print(f"customer_id uniques        : {n_ids}")
print(f"customer_unique_id uniques : {n_unique}")
print(f"Différence                 : {diff}")

if diff > 0:
    print(f"\n⚠️  {diff} customer_id correspondent au même client physique")
    print(f"   → Utiliser customer_unique_id pour le RFM dans 05_feature_engineering")
else:
    print(f"\n✓ Pas de doublon client")

print(f"\n--- Distribution customer_state (top 10) ---")
print(customers['customer_state'].value_counts().head(10))


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 11 — AUDIT SPÉCIFIQUE : geolocation")
print("   Problème cardinalité — plusieurs coords par zip")
print("="*60 + "\n")
# -------------------------------------------------------

n_total    = len(geo)
n_zip      = geo['geolocation_zip_code_prefix'].nunique()
ratio_geo  = n_total / n_zip

print(f"Lignes totales     : {n_total:>10}")
print(f"Zip codes uniques  : {n_zip:>10}")
print(f"Ratio moyen        : {ratio_geo:>10.1f} coordonnées par zip")
print(f"\n⚠️  Jointure directe → multiplication des lignes par {ratio_geo:.0f}")
print(f"   → Dédupliquer par zip (médiane) dans 03_data_support")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 12 — RAPPORT DE DÉCISION FINAL")
print("   Synthèse de l'audit — plan d'action pour cleaning")
print("="*60 + "\n")
# -------------------------------------------------------

decisions = [
    ("orders",    "Convertir dates",            "02_data_cleaning"),
    ("orders",    "Supprimer 8 delivered/null", "02_data_cleaning"),
    ("orders",    "order_status → category",    "02_data_cleaning"),
    ("items",     "shipping_limit_date → date", "02_data_cleaning"),
    ("items",     "IQR sur price + freight",    "02_data_cleaning"),
    ("items",     "Flag fret > prix",           "02_data_cleaning"),
    ("payments",  "payment_type → category",    "02_data_cleaning"),
    ("payments",  "Flag 'not_defined'",         "02_data_cleaning"),
    ("reviews",   "Dates → datetime",           "02_data_cleaning"),
    ("reviews",   "Nulls commentaires → OK",    "02_data_cleaning"),
    ("products",  "610 catégories → unknown",   "02_data_cleaning"),
    ("products",  "Dimensions → médiane",       "02_data_cleaning"),
    ("products",  "Merge translation EN",       "02_data_cleaning"),
    ("customers", "customer_id vs unique_id",   "03_data_support"),
    ("customers", "zip_code → string",          "03_data_support"),
    ("geo",       "Dédupliquer par zip",        "03_data_support"),
    ("sellers",   "seller_state → category",    "03_data_support"),
]

print(f"{'Table':<12} | {'Action':<35} | {'Notebook'}")
print("-" * 65)
for table, action, notebook in decisions:
    print(f"  {table:<12} | {action:<35} | {notebook}")

print(f"\n✓ Audit terminé — {len(decisions)} actions identifiées")
print(f"✓ Aucune donnée modifiée dans ce notebook")
print(f"\n→ Exécuter ensuite : 02_data_cleaning.ipynb")
print(f"→ Puis             : 03_data_support.ipynb")


   ÉTAPE 1 — CHARGEMENT DES TABLES BRUTES
   Source : data/raw/ — lecture seule

Table           |   Lignes | Colonnes
----------------------------------------
  orders          |    99441 |        8
  items           |   112650 |        7
  payments        |   103886 |        5
  reviews         |    99224 |        7
  products        |    32951 |        9
  customers       |    99441 |        5
  sellers         |     3095 |        4
  geo             |  1000163 |        5
  transl          |       71 |        2

✓ Toutes les tables chargées — aucune modification

   ÉTAPE 2 — AUDIT STRUCTURE : TOUTES LES TABLES
   Cours 3 §2 — identifier les types de données


  TABLE : ORDERS  — 99441 lignes × 8 colonnes
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
o